&nbsp;

## ChIP-Atlas Diff Analysis Parse Notebook

- 既に `diff Analysis` を実行済みだった heat-stress related ATAC-seq dataset に対し、`bedtools intersect` を行い、その結果をparse するための notebook

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import polars as pl
from IPython.display import display


os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### この notebook を実行するために必要な手順

1. [ChIP-Atlas Diff Analysis](https://chip-atlas.org/diff_analysis) を実行する
    1. 事前に定義した heat-stress (dataset A) vs. ctrl (dataset B) のデータに従ってペアを作成し、実行する
2. `bedtools intersect -wo`を実行する CWLファイル [../../Workflow/bedtools_intersect_v2_scatter.cwl](../../Workflow/bedtools_intersect_v2_scatter.cwl)を実行する

&nbsp;

```bash
cwltool --outdir ./out/ChIP-Atlas_diff/ATACseq/bedfile_intersect_v2/ \
./Workflow/bedtools_intersect_v2_scatter.cwl \
./config/bedtools_intersect/chipatlas_diff_atacseq_v2.yml
```

&nbsp;

#### WARNING

- 以下の処理では、q-value < 0.01 でフィルタリングしている

In [2]:
INTERSECT_DIFF_COLUMNS = [
    # A: ChIP-Atlas diff DPR (8 cols)  <- query (-a)
    "dpr_chrom", "dpr_start", "dpr_end",
    "norm_count_A", "norm_count_B",  # dataset A → Heat stress, B → Control
    "log2fc", "p_value", "q_value",
    # B: fanta.bio CRE (11 cols)       <- database (-b)
    "cre_chrom", "cre_start", "cre_end", "cre_id",
    "cre_score", "cre_strand", "thickStart", "thickEnd",
    "itemRgb", "cre_name", "pooled_cre_scores",
    # -wo
    "overlap_bp",
]

In [3]:
#繰り返し処理を行う
BED_INTERSECT_DIR = pathlib.Path("../../out/ChIP-Atlas_diff/ATACseq/bedfile_intersect_v2")

def read_intersect_bed(bed_path: pathlib.Path, q_threshold: float = 0.01) -> pl.DataFrame:
    """
    Read one bedtools intersect -wo output and tag with diff dataset ID from filename.
    default q-value threshold is 0.01.
    """
    # v2 filename: <dataset>_intersecting_<cre-peaks>.bed
    diff_dataset = bed_path.stem.split("_intersecting_", 1)[0]

    return (
        pl.read_csv(
            bed_path,
            separator="\t",
            has_header=False,
            new_columns=INTERSECT_DIFF_COLUMNS,
            # Force float dtypes: leading "1" q-values can make Polars infer Int64
            # and drop scientific-notation significant hits under ignore_errors.
            schema_overrides={
                "dpr_start": pl.Int64,
                "dpr_end": pl.Int64,
                "log2fc": pl.Float64,
                "p_value": pl.Float64,
                "q_value": pl.Float64,
                "cre_start": pl.Int64,
                "cre_end": pl.Int64,
                "cre_score": pl.Int64,
                "thickStart": pl.Int64,
                "thickEnd": pl.Int64,
                "overlap_bp": pl.Int64,
            },
            n_threads=16,
            ignore_errors=True,
        )
        .filter(pl.col("q_value") < q_threshold)
        .with_columns(
            # Fraction of CRE length covered by the DPR overlap (-wo)
            (pl.col("overlap_bp") / (pl.col("cre_end") - pl.col("cre_start"))).alias("overlap_frac_of_fantabio_cre"),
            pl.struct(
                "dpr_start",
                "dpr_end",
                "dpr_chrom"
            ).alias("dpr_interval")
        )
        .with_columns(
            pl.col("pooled_cre_scores").str.extract(r"directionality:([^|]+)", 1).cast(pl.Float64, strict=False).alias("directionality"),
            pl.col("pooled_cre_scores").str.extract(r"class:([^|]+)", 1).alias("fantabio_cre_class"),
        )
        .with_columns(
            pl.lit(diff_dataset).alias("diff_dataset"),
            pl.lit(bed_path.name).alias("source_bed_file"),
        )
        .drop("cre_score", "cre_strand", "thickStart", "thickEnd", "itemRgb", "pooled_cre_scores", "norm_count_A", "norm_count_B", "dpr_start", "dpr_end", "dpr_chrom")
    )

bed_paths = sorted(
    BED_INTERSECT_DIR.glob("*_intersecting_*.bed")
)

all_diff_df = (
    pl.concat(
        [read_intersect_bed(p) for p in bed_paths],
        how="vertical_relaxed",
    )
    .sort(
        "cre_chrom",
        descending=False,
    )
)


# mismatch = all_diff_df.filter(pl.col("cre_chrom") != pl.col("dpr_chrom"))
# print(f"mismatch rows: {mismatch.height}")
pl.Config.set_tbl_rows(10)
pl.Config.set_fmt_str_lengths(100)
display(all_diff_df)

log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
1.346413,5.4291e-53,7.6313e-50,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",12,0.0625,"{629141,629153,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.363021,2.5793e-49,3.1990e-46,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",88,0.458333,"{629153,629241,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.520449,1.7373e-20,4.2758e-18,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",22,0.114583,"{629241,629263,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.470083,4.9866e-9,2.4993e-7,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",5,0.026042,"{629263,629268,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.510948,3.2981e-17,6.1358e-15,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",5,0.026042,"{629268,629276,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.782456,0.000625,0.005354,"""chrY""",26670966,26671518,"""FCHS_447308""","""FCHS_447308""",184,0.333333,"{26670970,26671154,""chrY""}",-0.979798,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
0.966208,0.000259,0.004473,"""chrY""",11329762,11329941,"""FCHS_447044""","""FCHS_447044""",73,0.407821,"{11329868,11330053,""chrY""}",-0.884058,"""ELA""","""GSE101731_atac_hff_44C_60min""","""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.249991,0.000216,0.003874,"""chrY""",19077163,19077775,"""FCHS_447246""","""cp8@TTTY14""",99,0.161765,"{19077100,19077262,""chrY""}",-0.998313,"""PLA""","""GSE101731_atac_hff_44C_60min""","""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


&nbsp;

&nbsp;

&nbsp;

# Selection of Differencially Peaked Region (DPR)

&nbsp;

## (1) Filtering Condition

- `log2(Fold Change) > 0.5`
- [script/diff-analysis/diff_analysis_for_ChIP_ATAC_DNase.sh](https://github.com/inutano/chip-atlas/blob/master/script/diff-analysis/diff_analysis_for_ChIP_ATAC_DNase.sh)

In [4]:
all_diff_up = all_diff_df.filter(
    pl.col("log2fc") > 0.5
)

display(all_diff_up)

log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
1.346413,5.4291e-53,7.6313e-50,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",12,0.0625,"{629141,629153,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.363021,2.5793e-49,3.1990e-46,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",88,0.458333,"{629153,629241,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.520449,1.7373e-20,4.2758e-18,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",22,0.114583,"{629241,629263,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.470083,4.9866e-9,2.4993e-7,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",5,0.026042,"{629263,629268,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.510948,3.2981e-17,6.1358e-15,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",5,0.026042,"{629268,629276,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3.001242,0.000002,0.001012,"""chrX""",147981084,147981545,"""FCHS_593427""","""cp2@FMR1NB""",259,0.561822,"{147981286,147981595,""chrX""}",0.997951,"""PLA""","""GSE209675_mcf7_42C_60min""","""GSE209675_mcf7_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
0.895061,0.000014,0.000234,"""chrY""",11297628,11297795,"""FCHS_593898""","""FCHS_593898""",59,0.353293,"{11297469,11297687,""chrY""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
0.782456,0.000625,0.005354,"""chrY""",26670966,26671518,"""FCHS_447308""","""FCHS_447308""",184,0.333333,"{26670970,26671154,""chrY""}",-0.979798,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## (2) Filter by Overlap Fraction

- 対象は直前の `log2fc > 0.5` 通過後（`all_diff_up`）
- 同一 `diff_dataset` × `cre_id` で複数 DPR が重なる場合、`overlap_frac_of_fantabio_cre` が最大の行のみ残す
- check: `FCHS_5` の before / after を表示


In [5]:
# Among upregulated DPRs (log2fc > 0.5), keep max CRE-cover fraction
# per (diff_dataset, cre_id). Ties keep multiple rows.
CHECK_CRE_ID = "FCHS_5"

fchs5_before = all_diff_up.filter(pl.col("cre_id") == CHECK_CRE_ID).sort(
    "diff_dataset",
    "overlap_frac_of_fantabio_cre",
    descending=[False, True],
)
print(f"[before] {CHECK_CRE_ID}: {fchs5_before.height} rows")
display(fchs5_before)

all_diff_up_filtered = all_diff_up.filter(
    pl.col("overlap_frac_of_fantabio_cre")
    == pl.col("overlap_frac_of_fantabio_cre").max().over("diff_dataset", "cre_id")
)

fchs5_after = all_diff_up_filtered.filter(pl.col("cre_id") == CHECK_CRE_ID).sort(
    "diff_dataset",
    "overlap_frac_of_fantabio_cre",
    descending=[False, True],
)
print(
    f"[after]  {CHECK_CRE_ID}: {fchs5_after.height} rows "
    f"(all_diff_up {all_diff_up.height} -> {all_diff_up_filtered.height})"
)
display(fchs5_after)

[before] FCHS_5: 9 rows


log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
1.363021,2.5793e-49,3.1990e-46,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",88,0.458333,"{629153,629241,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.520449,1.7373e-20,4.2758e-18,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",22,0.114583,"{629241,629263,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.346413,5.4291e-53,7.6313e-50,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",12,0.0625,"{629141,629153,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.470083,4.9866e-9,2.4993e-7,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",5,0.026042,"{629263,629268,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.510948,3.2981e-17,6.1358e-15,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",5,0.026042,"{629268,629276,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
0.971139,2.8597e-25,3.1055e-22,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",86,0.447917,"{629151,629237,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_60min""","""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.074398,5.8290e-10,8.0810e-8,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",22,0.114583,"{629241,629263,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_60min""","""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.021146,0.00015,0.002939,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",5,0.026042,"{629263,629268,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_60min""","""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
0.990666,3.6290e-7,0.000022,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",5,0.026042,"{629268,629276,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_60min""","""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


[after]  FCHS_5: 2 rows (all_diff_up 8536 -> 6851)


log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
1.363021,2.5793e-49,3.1990e-46,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",88,0.458333,"{629153,629241,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
0.971139,2.8597e-25,3.1055e-22,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",86,0.447917,"{629151,629237,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_60min""","""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


&nbsp;

&nbsp;

&nbsp;

## After Filtering

In [6]:
display(all_diff_up_filtered)

log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
1.363021,2.5793e-49,3.1990e-46,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",88,0.458333,"{629153,629241,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.231145,3.7728e-53,5.4236e-50,"""chr1""",629469,630126,"""FCHS_6""","""cp1@MTND2P28""",549,0.835616,"{629276,630018,""chr1""}",0.999887,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.516768,1.5953e-65,5.3109e-62,"""chr1""",629938,630317,"""FCHS_7""","""cp2@MTND2P28""",82,0.216359,"{630235,630453,""chr1""}",0.771711,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.516768,1.5953e-65,5.3109e-62,"""chr1""",630319,630524,"""FCHS_447418""","""FCHS_447418""",134,0.653659,"{630235,630453,""chr1""}",-1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.250148,5.5115e-46,5.3634e-43,"""chr1""",631621,631968,"""FCHS_447420""","""cp4@LOC101928626""",86,0.247839,"{631882,632303,""chr1""}",0.565217,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3.001242,0.000002,0.001012,"""chrX""",147981084,147981545,"""FCHS_593427""","""cp2@FMR1NB""",259,0.561822,"{147981286,147981595,""chrX""}",0.997951,"""PLA""","""GSE209675_mcf7_42C_60min""","""GSE209675_mcf7_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
0.895061,0.000014,0.000234,"""chrY""",11297628,11297795,"""FCHS_593898""","""FCHS_593898""",59,0.353293,"{11297469,11297687,""chrY""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
0.782456,0.000625,0.005354,"""chrY""",26670966,26671518,"""FCHS_447308""","""FCHS_447308""",184,0.333333,"{26670970,26671154,""chrY""}",-0.979798,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


In [7]:
# Aggregate upregulated + max-overlap DPRs to CRE level
all_diff_up_select = all_diff_up_filtered.group_by(
    [
        "cre_chrom",
        "cre_start",
        "cre_end",
        "cre_id",
        "cre_name",
        # "dpr_chrom",
        "directionality",
        "fantabio_cre_class"
    ]
).agg(
    pl.col("overlap_frac_of_fantabio_cre").alias("overlap_frac_of_fantabio_cre"),
    pl.col("dpr_interval").alias("dpr_interval"),
    pl.col("log2fc").alias("log2fc"),
    pl.col("p_value").alias("p_value"),
    pl.col("q_value").alias("q_value"),
    pl.col("diff_dataset").unique().alias("diff_dataset"),
    pl.col("diff_dataset").unique().count().alias("diff_dataset_count"),
    pl.col("source_bed_file").alias("source_bed_file"),
).sort(
    "cre_chrom",
    "cre_start",
    "cre_end",
    descending=False
)

pl.Config.set_tbl_rows(6)
display(all_diff_up_select)

cre_chrom,cre_start,cre_end,cre_id,cre_name,directionality,fantabio_cre_class,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,str,str,f64,str,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",1.0,"""ELA""","[0.458333, 0.447917]","[{629153,629241,""chr1""}, {629151,629237,""chr1""}]","[1.363021, 0.971139]","[2.5793e-49, 2.8597e-25]","[3.1990e-46, 3.1055e-22]","[""GSE101731_atac_hff_44C_120min"", ""GSE101731_atac_hff_44C_60min""]",2,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chr1""",629469,630126,"""FCHS_6""","""cp1@MTND2P28""",0.999887,"""PLA""","[0.835616, 0.835616]","[{629276,630018,""chr1""}, {629276,630018,""chr1""}]","[1.231145, 0.92496]","[3.7728e-53, 4.6591e-30]","[5.4236e-50, 9.1072e-27]","[""GSE101731_atac_hff_44C_120min"", ""GSE101731_atac_hff_44C_60min""]",2,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chr1""",629938,630317,"""FCHS_7""","""cp2@MTND2P28""",0.771711,"""PLA""","[0.216359, 0.216359]","[{630235,630453,""chr1""}, {630235,630452,""chr1""}]","[1.516768, 1.082735]","[1.5953e-65, 6.4854e-31]","[5.3109e-62, 1.3114e-27]","[""GSE101731_atac_hff_44C_120min"", ""GSE101731_atac_hff_44C_60min""]",2,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",11329762,11329941,"""FCHS_447044""","""FCHS_447044""",-0.884058,"""ELA""",[0.407821],"[{11329868,11330053,""chrY""}]",[0.966208],[0.000259],[0.004473],"[""GSE101731_atac_hff_44C_60min""]",1,"[""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chrY""",19077163,19077775,"""FCHS_447246""","""cp8@TTTY14""",-0.998313,"""PLA""",[0.161765],"[{19077100,19077262,""chrY""}]",[1.249991],[0.000216],[0.003874],"[""GSE101731_atac_hff_44C_60min""]",1,"[""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chrY""",26670966,26671518,"""FCHS_447308""","""FCHS_447308""",-0.979798,"""ELA""",[0.333333],"[{26670970,26671154,""chrY""}]",[0.782456],[0.000625],[0.005354],"[""GSE101731_atac_hff_44C_120min""]",1,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"


In [8]:
all_diff_up_select.write_parquet(
    "../../out/ChIP-Atlas_diff/diff_analysis_filter_parquet/only_cre/fantabio_cre_list_diff_analysis_atac.parquet"
)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Intergration (upregulated transcripts)

In [9]:
# up5p transcript lists + CRE
fantabio_HN2_up5p_transcripts_cre = pl.read_parquet(
    "../../out/fantabio/parquet_file/up5p_transcript_fantabio_v261001.parquet"
).join(
    all_diff_up_select,
    on=["cre_id", "cre_chrom", "cre_name", "directionality", "fantabio_cre_class"],
    how="inner",
    coalesce=True
)

fantabio_HN2_up5p_transcripts_cre.write_parquet(
    "../../out/ChIP-Atlas_diff/diff_analysis_filter_parquet/transcript_cre/fantabio_HN2_up5p_v261002_diff_analysis_atac.parquet"
)

print(f"up5p_fantabio transcript count: {fantabio_HN2_up5p_transcripts_cre.select('transcript_id').n_unique()}")
print(f"up5p_fantabio transcript-CRE pairs count: {fantabio_HN2_up5p_transcripts_cre.select(['transcript_id', 'cre_id']).n_unique()}")
pl.Config.set_tbl_rows(4)
display(fantabio_HN2_up5p_transcripts_cre)

up5p_fantabio transcript count: 2497
up5p_fantabio transcript-CRE pairs count: 4423


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red""",161523744,161524487,"[0.296097, 0.117093, 0.351279]","[{161524223,161524443,""chr1""}, {161524400,161524568,""chr1""}, {161524226,161524581,""chr1""}]","[1.209197, 1.832055, 1.892709]","[2.7404e-8, 1.8112e-16, 7.5870e-7]","[0.000001, 2.0457e-12, 0.000487]","[""GSE101731_atac_hff_44C_120min"", ""GSE173536_atac_d21_42C_60min"", ""GSE209675_mcf7_42C_60min""]",3,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE173536_atac_d21_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE209675_mcf7_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28106""","""chr1""",161524161,"""161524546""","""cp4@HSPA6""","[""ENST00000309758.6"", ""NM_002155.5"", … ""AK295212""]","""0""",0,"[{""EH38E2843855"",""PLS""}]","[{""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX3322007"",560}]}, {""CTCF"",3651,[{""SRX10975999"",3651}, {""SRX10976007"",2920}, … {""SRX13551280"",511}]}, … {""MED12"",511,[{""SRX10852993"",511}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]","[{""hg19::chr1:161494328..161494332,+;hg_6264.1"",""p1@HSPA6""}]",null,0.943234,"""PLA""","""204,0,0""","""red""",161524160,161524546,"[0.569948, 0.378238, … 0.829016]","[{161524223,161524443,""chr1""}, {161524400,161524568,""chr1""}, … {161524226,161524581,""chr1""}]","[1.209197, 1.832055, … 1.892709]","[2.7404e-8, 1.8112e-16, … 7.5870e-7]","[0.000001, 2.0457e-12, … 0.000487]","[""GSE101731_atac_hff_44C_120min"", ""GSE173536_atac_d21_42C_60mi

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Protein-coding Transcripts

In [14]:
# (check whether the protein-coding transcripts - CREs are heat stress-responsive related transcripts)
fantabio_HN2_up5p_transcripts_cre_pc = fantabio_HN2_up5p_transcripts_cre.filter(
    pl.col("transcript_type") == "protein_coding"
)

up5p_pc_list = fantabio_HN2_up5p_transcripts_cre_pc.select("transcript_id").unique().write_csv(
    "../../out/metascape/transcript_list/up5p/HN2_up5p_fantabio_check_diff_analysis_atac_pc_v261002.tsv",
    separator="\t",
    include_header=False
)

print(f"up5p_fantabio protein-coding transcript count: {fantabio_HN2_up5p_transcripts_cre_pc.select("transcript_id").n_unique()}")
print(f"up5p_fantabio protein-coding transcript-CRE pairs count: {fantabio_HN2_up5p_transcripts_cre_pc.select(['transcript_id', 'cre_id']).n_unique()}")
display(fantabio_HN2_up5p_transcripts_cre_pc)

up5p_fantabio protein-coding transcript count: 1667
up5p_fantabio protein-coding transcript-CRE pairs count: 3018


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red""",161523744,161524487,"[0.296097, 0.117093, 0.351279]","[{161524223,161524443,""chr1""}, {161524400,161524568,""chr1""}, {161524226,161524581,""chr1""}]","[1.209197, 1.832055, 1.892709]","[2.7404e-8, 1.8112e-16, 7.5870e-7]","[0.000001, 2.0457e-12, 0.000487]","[""GSE101731_atac_hff_44C_120min"", ""GSE173536_atac_d21_42C_60min"", ""GSE209675_mcf7_42C_60min""]",3,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE173536_atac_d21_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE209675_mcf7_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28106""","""chr1""",161524161,"""161524546""","""cp4@HSPA6""","[""ENST00000309758.6"", ""NM_002155.5"", … ""AK295212""]","""0""",0,"[{""EH38E2843855"",""PLS""}]","[{""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX3322007"",560}]}, {""CTCF"",3651,[{""SRX10975999"",3651}, {""SRX10976007"",2920}, … {""SRX13551280"",511}]}, … {""MED12"",511,[{""SRX10852993"",511}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]","[{""hg19::chr1:161494328..161494332,+;hg_6264.1"",""p1@HSPA6""}]",null,0.943234,"""PLA""","""204,0,0""","""red""",161524160,161524546,"[0.569948, 0.378238, … 0.829016]","[{161524223,161524443,""chr1""}, {161524400,161524568,""chr1""}, … {161524226,161524581,""chr1""}]","[1.209197, 1.832055, … 1.892709]","[2.7404e-8, 1.8112e-16, … 7.5870e-7]","[0.000001, 2.0457e-12, … 0.000487]","[""GSE101731_atac_hff_44C_120min"", ""GSE173536_atac_d21_42C_60mi

In [12]:
# check HSPH1
HSPH1 = fantabio_HN2_up5p_transcripts_cre_pc.filter(
    pl.col("gene_symbol") == "HSPH1"
)

display(HSPH1)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000954002.1""",276,13,117,263,174,9,68,165,"""HSPH1""","""HGNC:16969""","""protein_coding""","""ENSG00000120694.21""","[""basic"", ""TAGENE"", ""CCDS""]","""-""",3,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""ENCODE_tissue""]",1,"""FCHS_110516""","""chr13""",31161745,"""31162676""","""cp7@HSPH1""","[""ENST00000954002.1"", ""ENST00000320027.10"", … ""AK311464""]","""0""",0,"[{""EH38E3059049"",""pELS""}, {""EH38E3059050"",""PLS""}, … {""EH38E3059052"",""pELS""}]","[{""RFX1"",4409,[{""SRX2423398"",4409}, {""SRX2424673"",4103}, … {""SRX5457514"",596}]}, {""HSF1"",3960,[{""SRX16646095"",3960}, {""SRX16646102"",3724}, … {""SRX13845688"",503}]}, … {""ZHX2"",507,[{""SRX3636887"",507}]}]","[{""HS_00184065"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, {""HS_00184066"",""H1 BMP4 Derived Trophoblast Cultured Cells"",""Trophoblast cell"",""Others"",""SVM""}, … {""HS_00184072"",""Fetal Intestine Small"",""Small intestine"",""Intestine"",""SVM""}]","[{""hg19::chr13:31735993..31736009,-;hg_136690.1"",""p3@HSPH1""}, {""hg19::chr13:31736027..31736053,-;hg_136691.1"",""p1@HSPH1""}, … {""hg19::chr13:31736780..31736803,+;hg_134654.1"",""hg_134654.1""}]",null,-0.981672,"""PLA""","""0,0,204""","""blue""",31161744,31162676,"[0.05794, 0.040773]","[{31161798,31161852,""chr13""}, {31161701,31161782,""chr13""}]","[1.492452, 1.617098]","[5.5610e-10, 2.6045e-10]","[0.000003, 6.8502e-7]","[""GSE173536_atac_d21_42C_60min"", ""GSE173536_atac_t21_42C_60min""]",2,"[""GSE173536_atac_d21_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE173536_atac_t21_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000380405.7""",243,17,146,226,131,8,112,123,"""HSPH1""","""HGNC:16969""","""protein_coding""","""ENSG00000120694.21""","[""basic"", ""GENCODE_Primary"", … ""CCDS""]","""-""",4,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_482632""","""chr13""",31161036,"""31161469""","""cp8@HSPH1""","[""CU691504""]","""-113""",-458,"[{""EH38E3059046"",""pELS""}, {""EH38E3059047"",""pELS""}, {""EH38E3059048"",""pELS""}]","[{""CTCF"",2420,[{""SRX10975999"",2420}, {""SRX12008355"",2273}, … {""SRX13551281"",758}]}, {""NELFCD"",1685,[{""SRX15147204"",1685}]}, … {""KAT7"",530,[{""SRX101303"",530}]}]","[{""HS_00184065"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, {""HS_00184066"",""H1 BMP4 Derived Trophoblast Cultured Cells"",""Trophoblast cell"",""Others"",""SVM""}, … {""HS_00184072"",""Fetal Intestine Small"",""Small intestine"",""Intestine"",""SVM""}]","[{""hg19::chr13:31735232..31735251,-;hg_136689.1"",""p15@HSPH1""}]",null,0.5,"""ELA""","""255,202,74""","""yellow""",31161035,31161469,[0.317972],"[{31161079,31161217,""chr13""}]",[1.17786],[0.000013],[0.005395],"[""GSE173536_atac_t21_42C_60min""]",1,"[""GSE173536_atac_t21_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## lncRNA

In [15]:
fantabio_HN2_up5p_transcripts_cre_lncrna = fantabio_HN2_up5p_transcripts_cre.filter(
    pl.col("transcript_type") == "lncRNA"
).sort(
    pl.col([
        "HN2",
        "transcript_id"
    ]),
    descending=True
)

up5p_lncrna_list = fantabio_HN2_up5p_transcripts_cre_lncrna.select("transcript_id").unique().write_csv(
    "../../out/metascape/transcript_list/up5p/HN2_up5p_fantabio_check_diff_analysis_atac_lncrna_v261002.tsv",
    separator="\t",
    include_header=False
)

pl.Config.set_tbl_rows(4)
print(f"up5p_fantabio lncRNA transcript count: {fantabio_HN2_up5p_transcripts_cre_lncrna.select("transcript_id").n_unique()}")
print(f"up5p_fantabio lncRNA transcript-CRE pairs count: {fantabio_HN2_up5p_transcripts_cre_lncrna.select(['transcript_id', 'cre_id']).n_unique()}")
display(fantabio_HN2_up5p_transcripts_cre_lncrna)

up5p_fantabio lncRNA transcript count: 291
up5p_fantabio lncRNA transcript-CRE pairs count: 459


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000729765.1""",315,20,71,295,206,4,41,202,null,null,"""lncRNA""","""ENSG00000295391.1""","[""basic"", ""TAGENE""]","""+""",2,"""URS0002A422B0""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_110516""","""chr13""",31161745,"""31162676""","""cp7@HSPH1""","[""ENST00000954002.1"", ""ENST00000320027.10"", … ""AK311464""]","""0""",0,"[{""EH38E3059049"",""pELS""}, {""EH38E3059050"",""PLS""}, … {""EH38E3059052"",""pELS""}]","[{""RFX1"",4409,[{""SRX2423398"",4409}, {""SRX2424673"",4103}, … {""SRX5457514"",596}]}, {""HSF1"",3960,[{""SRX16646095"",3960}, {""SRX16646102"",3724}, … {""SRX13845688"",503}]}, … {""ZHX2"",507,[{""SRX3636887"",507}]}]","[{""HS_00184065"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, {""HS_00184066"",""H1 BMP4 Derived Trophoblast Cultured Cells"",""Trophoblast cell"",""Others"",""SVM""}, … {""HS_00184072"",""Fetal Intestine Small"",""Small intestine"",""Intestine"",""SVM""}]","[{""hg19::chr13:31735993..31736009,-;hg_136690.1"",""p3@HSPH1""}, {""hg19::chr13:31736027..31736053,-;hg_136691.1"",""p1@HSPH1""}, … {""hg19::chr13:31736780..31736803,+;hg_134654.1"",""hg_134654.1""}]",null,-0.981672,"""PLA""","""0,0,204""","""blue""",31161744,31162676,"[0.05794, 0.040773]","[{31161798,31161852,""chr13""}, {31161701,31161782,""chr13""}]","[1.492452, 1.617098]","[5.5610e-10, 2.6045e-10]","[0.000003, 6.8502e-7]","[""GSE173536_atac_d21_42C_60min"", ""GSE173536_atac_t21_42C_60min""]",2,"[""GSE173536_atac_d21_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE173536_atac_t21_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000702650.2""",300,17,89,283,194,6,51,188,null,null,"""lncRNA""","""ENSG00000290051.2""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""+""",4,"""URS0002A32179""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""Assembled from PacBio CLS3 reads"", ""CLS3 project""]",2,"""FCHS_353060""","""chr6""",28335283,"""28335951""","""cp7@ZSCAN31""","[""ENST00000917876.1"", ""ENST00000881658.1"", … ""AL833416""]","""0""",0,"[{""EH38E2457180"",""PLS""}, {""EH38E3698732"",""pELS""}, {""EH38E3698733"",""PLS""}]","[{""HSF1"",3747,[{""SRX16646111"",3747}, {""SRX6858250"",1860}, … {""SRX8512778"",662}]}, {""GATA1"",3265,[{""SRX10218286"",3265}, {""SRX10218289"",3222}, … {""SRX5931551"",511}]}, … {""FOXA2"",509,[{""SRX8089910"",509}]}]","[{""HS_00074454"",""Fetal Intestine Small"",""Small intestine"",""Intestine"",""SVM""}, {""HS_00074455"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, … {""HS_00074463"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]","[{""hg19::chr6:28303459..28303472,+;hg_66632.1"",""hg_66632.1""}, {""hg19::chr6:28303520..28303535,+;hg_66633.1"",""hg_66633.1""}]",null,0.855506,"""PLA""","""204,0,0""","""red""",28335282,28335951,[0.428999],"[{28335548,28335835,""chr6""}]",[0.983649],[3.0776e-9],[1.6168e-7],"[""GS

&nbsp;

&nbsp;

&nbsp;

## Other Transcripts

In [16]:
fantabio_HN2_up5p_transcripts_cre_other = fantabio_HN2_up5p_transcripts_cre.filter(
    (pl.col("transcript_type") != "lncRNA") & (pl.col("transcript_type") != "protein_coding")
).sort(
    pl.col("HN2"),
    descending=True
)

up5p_other_list = fantabio_HN2_up5p_transcripts_cre_other.select("transcript_id").unique().write_csv(
    "../../out/metascape/transcript_list/up5p/HN2_up5p_fantabio_check_diff_analysis_atac_other_v261002.tsv",
    separator="\t",
    include_header=False
)

pl.Config.set_tbl_rows(10)
print(fantabio_HN2_up5p_transcripts_cre_other.select("transcript_id").n_unique())
print(fantabio_HN2_up5p_transcripts_cre_other.group_by("transcript_type").agg(pl.col("transcript_id").n_unique().alias("transcript_type_count")))
display(fantabio_HN2_up5p_transcripts_cre_other)

539
shape: (10, 2)
┌────────────────────────────────────┬───────────────────────┐
│ transcript_type                    ┆ transcript_type_count │
│ ---                                ┆ ---                   │
│ str                                ┆ u32                   │
╞════════════════════════════════════╪═══════════════════════╡
│ nonsense_mediated_decay            ┆ 220                   │
│ vault_RNA                          ┆ 1                     │
│ miRNA                              ┆ 1                     │
│ retained_intron                    ┆ 168                   │
│ Mt_rRNA                            ┆ 1                     │
│ protein_coding_CDS_not_defined     ┆ 140                   │
│ processed_pseudogene               ┆ 2                     │
│ transcribed_unprocessed_pseudogene ┆ 2                     │
│ snRNA                              ┆ 1                     │
│ TEC                                ┆ 3                     │
└───────────────────────────────────

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000540260.1""",328,9,69,319,216,1,34,215,"""FKBP4""","""HGNC:3720""","""protein_coding_CDS_not_defined""","""ENSG00000004478.9""","[""non_canonical_TEC""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""non canonical TEC""]",1,"""FCHS_86240""","""chr12""",2794544,"""2795205""","""cp3@FKBP4,cp1@ITFG2-AS1""","[""ENST00000839341.1"", ""ENST00000839340.1"", … ""DQ893668""]","""0""",452,"[{""EH38E1587359"",""pELS""}, {""EH38E2998737"",""PLS""}]","[{""HSF1"",3307,[{""SRX16646111"",3307}, {""SRX16646095"",3215}, … {""SRX3322007"",507}]}, {""ONECUT2"",2305,[{""SRX3927334"",2305}, {""SRX3927332"",713}, … {""SRX2731587"",570}]}, … {""FLI1"",508,[{""SRX117011"",508}]}]",null,"[{""hg19::chr12:2903742..2903763,-;hg_128965.1"",""hg_128965.1""}, {""hg19::chr12:2903775..2903794,-;hg_128966.1"",""hg_128966.1""}, … {""hg19::chr12:2904305..2904316,+;hg_123449.1"",""p6@FKBP4""}]",null,0.996397,"""PLA""","""204,0,0""","""red""",2794543,2795205,"[0.246224, 0.311178, 0.178248]","[{2794999,2795162,""chr12""}, {2794858,2795064,""chr12""}, {2794737,2794855,""chr12""}]","[1.814999, 0.857491, 2.723118]","[3.3135e-11, 0.000012, 1.4179e-19]","[2.5591e-9, 0.009422, 2.1133e-15]","[""GSE101731_atac_hff_44C_120min"", ""GSE173536_atac_d21_42C_60min"", ""GSE173536_atac_t21_42C_60min""]",3,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE173536_atac_d21_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE173536_atac_t21_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000555662.1""",329,18,59,311,215,5,31,210,"""HSP90AA1""","""HGNC:5253""","""retained_intron""","""ENSG00000080824.20""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_131437""","""chr14""",102086202,"""102086823""","""cp12@HSP90AA1""","[""ENST00000554401.1"", ""BX247955"", ""BC108695""]","""0""",495,"[{""EH38E4115864"",""pELS""}, {""EH38E4115865"",""pELS""}, … {""EH38E4115867"",""pELS""}]","[{""CTCF"",2380,[{""SRX10975999"",2380}, {""SRX10976001"",2275}, … {""SRX14609196"",628}]}, {""NELFCD"",1954,[{""SRX15147204"",1954}]}, … {""Crotonyllysine"",507,[{""SRX19488269"",507}]}]","[{""HS_00200033"",""Foreskin Melanocyte Primary Cells skin01"",""Prepuce"",""Skin"",""SVM""}, {""HS_00200034"",""Pancreas"",""Pancreas"",""Pancreas"",""SVM""}]","[{""hg19::chr14:102552659..102552678,-;hg_145005.1"",""p19@HSP90AA1""}, {""hg19::chr14:102552762..102552776,-;hg_145006.1"",""p21@HSP90AA1""}, {""hg19::chr14:102552785..102552808,-;hg_145007.1"",""p37@HSP90AA1""}]",null,-0.884058,"""PLA""","""0,0,204""","""blue""",102086201,102086823,"[0.516077, 0.324759]","[{102086502,102086841,""chr14""}, {102086418,102086620,""chr14""}]","[1.809335, 1.703382]","[6.3003e-24, 1.9019e-16]","[1.2453e-19, 1.4578e-12]","[""GSE173536_atac_d21_42C_60min"", ""GSE173536_atac_t21_42C_60min""]",2,"[""GSE173536_atac_d21_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE173536_atac_t21_42C_60min_intersecting